In [1]:
import vtk
from pathlib import Path
from scipy.stats import multivariate_normal
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy import io as scio
from scipy import stats as sci_stats
import pickle
import scipy.spatial as sp
from typing import Union,Dict,List
from procrustes.generalized import generalized
from procrustes import generic 

In [22]:
#variables
path_to_processed_cohort = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/mouse_skull_workdir"

result_mean_mesh = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/average_nl_mesh.obj"

result_energies = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/deviation_energies.mat"

snapshot_meshes = r"/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/meshes_dict_snapshot.pkl"



read meshes

In [2]:
def compute_center_of_mesh(mesh):
    centerOfMassFilter = vtk.vtkCenterOfMass()
    centerOfMassFilter.SetInputData(mesh)
    centerOfMassFilter.SetUseScalarsAsWeights(False)
    centerOfMassFilter.Update();
    center = centerOfMassFilter.GetCenter()
    return center




reader = vtk.vtkOBJReader()
num_of_points = 0
meshes_dict = {}
for path in tqdm(list(Path(path_to_processed_cohort).iterdir())):
    mesh_file = path / "head_affine_registered.obj"
    if not mesh_file.exists():
        continue
    reader.SetFileName(mesh_file)
    reader.Update()
    mesh = reader.GetOutput()
    points = mesh.GetPoints()
    center_of_mesh = np.array( compute_center_of_mesh(mesh) )
    num_of_points = points.GetNumberOfPoints()
    res_mesh = []
    for i in range(points.GetNumberOfPoints()):
        pt = points.GetPoint(i)
        res_mesh.append(list( np.array( pt) - center_of_mesh ))
    meshes_dict[str( mesh_file.parent.name ) ] = res_mesh


NameError: name 'path_to_processed_cohort' is not defined

Analyse statistics

In [4]:
def write_obj_2_file(obj,filename):
    f = open(filename,"wb")
    pickle.dump(obj,f)
    f.close()

def compute_descriptive_statistics(points):
    mean = np.mean(points, axis=0)
    std_dev = np.std(points, axis=0)
    # Add any other statistics you're interested in
    return mean, std_dev

def compute_outliers(meshes_dict: Union[Dict[str,np.ndarray],List[np.ndarray]]):
    
    stats = []
    index_sum=None
    if isinstance(meshes_dict,list):
        num_of_points = len( meshes_dict[0] )
        working_dict = list(zip(range(num_of_points),meshes_dict))
        
    else:
        num_of_points = len( list(meshes_dict.values())[0] )
        working_dict = list(meshes_dict.items())
            
    for i in tqdm(range(num_of_points)): # i in range of num_of_points
        index = []
        points = []
        #print (i, working_dict)
        for key, mesh_v in working_dict:
            index.append(key)
            points.append(mesh_v[i])
        
        df = pd.DataFrame( data = points, index=index, columns=['x','y','z'])
        
        indexes = (np.abs(sci_stats.zscore(df)) < 2).all(axis=1)
        
        if index_sum is None:
            index_sum = (~indexes).astype(int)
        else:
            index_sum += (~indexes).astype(int)
        df = df[indexes]
        stats.append(df.describe())
    
    return stats, index_sum

    

stats, index_sum = compute_outliers(meshes_dict)



  0%|          | 0/25680 [00:00<?, ?it/s]

In [18]:
#### Generate meshes 
import copy

meshes_dict_copy = copy.copy(meshes_dict)


#copy dict 
for k,v in sorted(index_sum.items(), key=lambda x:x[1]):
    if v > 1000:
        meshes_dict_copy.pop(k)
#clear items
stats, index_sum = compute_outliers(meshes_dict_copy)


  0%|          | 0/25680 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [51]:
stds = [x.loc['std'].sum() for x in stats] # energies 

In [52]:
#mean mesh
#iterate over gaces 


def write_mesh(mesh_Reader, points, output_name: str):
    mesh = mesh_Reader.GetOutput()
    vtkpoints = vtk.vtkPoints()
    for x in points:
        vtkpoints.InsertNextPoint(x)
    mesh.SetPoints(vtkpoints) # add points /
    writer = vtk.vtkOBJWriter()
    writer.SetInputData(mesh)
    writer.SetFileName(output_name)
    writer.Write()

    
points = [ [x.loc['mean']['x'], x.loc['mean']['y'], x.loc['mean']['z'] ] for x in stats ]
vtkpoints = vtk.vtkPoints()

i = 0 
for x in points:
    vtkpoints.InsertNextPoint(x)
    # Compute the color from the color map based on the energy value

mesh = reader.GetOutput()
mesh.SetPoints(vtkpoints) # add points 
writer = vtk.vtkOBJWriter()
writer.SetInputData(mesh)
writer.SetFileName(result_mean_mesh)
writer.Write()



polygons = mesh.GetPolys()
faces = []
# Iterate over the polygons to extract the faces
polygons.InitTraversal()
while True:
    faceIds = vtk.vtkIdList()
    if polygons.GetNextCell(faceIds):
        face = [faceIds.GetId(i) for i in range(faceIds.GetNumberOfIds())]
        faces.append(face)
    else:
        break
        
### compute colors and write to mat 
energies = []
for x in faces:
    en = 0 
    for p in x:
        en+= stds[p]
    energies.append(en)

scio.savemat(result_energies, {"energies": energies})


In [ ]:
sorted(index_sum.items(), key=lambda x:x[1])


In [9]:
write_obj_2_file(meshes_dict_copy, snapshot_meshes)

In [ ]:
pd.DataFrame(stds).plot()

In [16]:
def procreustes_alignment(reference_mesh, compute_mesh):
    #("STARTED")
    matx1,mesh_aligned, translation = sp.procrustes(reference_mesh, compute_mesh)
    print(translation )
    return mesh_aligned

def gpa(meshes, max_iterations=4, threshold=1e-6):
    num_meshes = len(meshes)
    num_vertices = len(meshes[0])

    # Convert meshes to 3D numpy array
    stacked_meshes = np.stack(meshes)

    # Initialize reference shape as the first mesh
    reference_mesh = stacked_meshes[0]

    for iteration in tqdm(range(max_iterations)):

        # Step 1: Superimpose all meshes to the current reference shape
        aligned_meshes = [reference_mesh]

        # futures = [executor.submit(procreustes_alignment, reference_mesh, mesh) for mesh in stacked_meshes[1:]]
        #
        # for f in tqdm(futures):
        #     aligned_meshes.append( f.result() )

        for mesh in tqdm(stacked_meshes[1:]):
            aligned_meshes.append( procreustes_alignment(reference_mesh, mesh) )
        # Step 2: Compute the mean shape of the current set of superimposed shapes
        mean_mesh = np.mean(np.stack(aligned_meshes), axis=0)
        #print("mean mesh shape", mean_mesh.shape)
        # Step 3: Check for convergence based on Procrustes distance
        _, _, procrustes_distance = sp.procrustes(reference_mesh, mean_mesh)
        print("procrustes distance ", procrustes_distance)
        if procrustes_distance < threshold:
            break

        # Update the reference shape
        reference_mesh = mean_mesh

    return reference_mesh, aligned_meshes

In [32]:
ref_mesh, aligned_meshes= gpa(list(meshes_dict.values()),max_iterations=100, threshold=1e-10)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/760 [00:00<?, ?it/s]

0.0007117631533120815
0.0019395718199111962
0.0013557544505211926
0.010666975683683376
0.000528934528075562
0.0008498255018115558
0.0008786153482626336
0.0017212492891473185
0.0011894655376393421
0.0007840034035494504
0.0009156381146294461
0.000595868495119712
0.0006825345465383627
0.0007131429879020208
0.0012238172583893844
0.0027738732936776873
0.0010984212808238423
0.0007373855830002701
0.0006145150191531158
0.0005851166303868928
0.0015995128262624744
0.0012006775814702724
0.000775173098302866
0.0007426563058583315
0.0012291064370528496
0.011329782974711478
0.0010905299180847309
0.0030342312233318037
0.3692797759368038
0.001062570509960846
0.0007060088018407088
0.0008245139614488276
0.0009179724678998385
0.0009278760512609412
0.000772333077371143
0.000871523221692174
0.0006034115707402485
0.001503677320929751
0.0012105499411486837
0.7962268914406276
0.0009989456847252914
0.001154444145720599
0.0007913547058323801
0.0009401722908121843
0.0009773408336255203
0.0009535485407235808
0.00

0.00079572633534507
0.0009250221717497494
0.0009296516311589877
0.0014948305716450077
0.00116661718149153
0.0012304339666882355
0.0009371691090806231
0.0008216197468362019
0.0004946494618473531
0.0009990022058766798
0.0015221820143381734
0.0009704611062434916
0.002691089871994734
0.0008610527360913873
0.0008437503447638937
0.003923874753348615
0.0011110376432259504
0.0006657358358215056
0.0009480421564643855
0.0012558989496124784
0.0012763575656441696
0.001018907627089273
0.001421062110426313
0.000741500076301587
0.0010384970915965214
0.0006236631079433591
0.0009927603242676501
0.0010848596079422609
0.0011165435181428515
0.0005386355535010543
0.0006569656227975634
0.0008621918710901301
0.0013988473382622348
0.0014919491605239414
0.00088387086307232
0.0012077042325026623
0.0015621437597517869
0.0009365362019743461
0.0004158089359973303
0.0007870313004724863
0.002355837134893039
0.0012592940354078572
0.0007238882715352784
0.0018045868065553197
0.0014358350797489773
0.0008423840144462676


  0%|          | 0/760 [00:00<?, ?it/s]

...

procrustes distance  3.650178327033816e-09


  0%|          | 0/760 [00:00<?, ?it/s]

0.00029465758544745157
0.001569396752466326
0.0009105762836633793
0.009722439626025892
0.00038224059782262934
0.0006218774340904894
0.0005373607760154898
0.0013118658346775586
0.0007616425448072764
0.00045585874981582316
0.00038785363646226794
0.0004165966084022632
0.000601743955418647
0.00046899939024919364
0.0005181484867216486
0.00218994978079004
0.0006901485412725751
0.0005392049328743564
0.0005370994388258317
0.00036442979454231545
0.0011791294554876716
0.0007676695196183467
0.0005226617788395323
0.00038749328375198993
0.0006194250172557122
0.010389668528733957
0.0005985470694711878
0.0026141755467416305
0.3644971800466529
0.0006321471712053563
0.0004065763303916527
0.000422698859179748
0.0003624081365957152
0.000549097622927537
0.00047315230497130766
0.00039137990578916554
0.00036453357466660803
0.0007148778324983673
0.000569579509475397
0.7942910109779248
0.0007510698154976986
0.0006938147872983117
0.0003844109625500884
0.0004660698564022007
0.0005155154471243811
0.0006675459970

0.0006087517301168498
0.0008087251281162478
0.0010972244770179793
0.0010291664620882407
0.0005617189958561826
0.0007755042075338421
0.0022915795136036845
0.0007619273786499873
0.0007015153447943465
0.0006007754174979546
0.0006224019515779679
0.0005771216971113918
0.0007346264428253477
0.0004095927810151608
0.001002563496457647
0.00045408520597151584
0.0004997842291244652
0.0003922049007962891
0.0006416299321832094
0.0007422328321051101
0.0005155436307330026
0.002082969406813105
0.00039228991420358966
0.000580385335528806
0.003414258906163504
0.0008392870588285107
0.0003744680009428288
0.0005784114300466954
0.000730347049062755
0.000774459664852717
0.0006544520286913922
0.000734275716174048
0.00034290789912918207
0.0006662543041916856
0.0005188308679187499
0.0011019362580167791
0.00043962004224481935
0.0006571375596643543
0.0004051472849945064
0.00034587368391658034
0.0004902427685499862
0.0008504708826205139
0.0012244695624017755
0.0007630972536839962
0.0005836070803983626
0.0009003116

procrustes distance  6.242013624970626e-14


In [17]:
ref_mesh.tolist()

[[-0.001393040647410148, -0.0008015904911703611, -0.009540548125661534],
 [-0.0015416400606720153, -0.0007260539697276672, -0.009523983835605175],
 [0.0018834233928237566, -0.0011775447863964829, -0.009287038011611926],
 [-0.0012917123362519618, -0.0006168927700519111, -0.009528734025789982],
 [-0.0014390330918047588, -0.0005293343308015382, -0.009519549299221492],
 [-0.001390553994247837, -0.0003409274356122915, -0.009501013503156273],
 [0.002211092447378296, -0.001104705466298391, -0.009290958445282778],
 [0.0023613072626125628, -0.0012274725528023485, -0.009236406542605479],
 [0.0018847331980844848, -0.000833598234527254, -0.009374709834580884],
 [-0.0011784896313543991, -0.00018186322084098232, -0.009485778149427066],
 [0.001836812526832397, -0.0005686644991228651, -0.009397232651763881],
 [0.001829191946057237, -0.00035513918378640686, -0.009402937658652329],
 [0.002060980268946745, -0.0004582652121898643, -0.009332391606407489],
 [0.001716083119371921, 0.00023321171760585174, -0.

In [22]:
write_mesh(reader,points=ref_mesh.tolist(), output_name=result_mean_mesh)

  0%|          | 0/25680 [00:00<?, ?it/s]

0      True
1      True
2      True
3      True
4      True
       ... 
473    True
474    True
475    True
476    True
477    True
Length: 478, dtype: bool
[]


ValueError: operands could not be broadcast together with shapes (478,) (0,) 

In [30]:
mes,a = generalized([np.array(x) for x in meshes_dict.values()],tol=1e-11)

In [40]:
mes[0],aligned_meshes[0],list(meshes_dict.values())[0]

(array([[ -1.69232971,  -1.1433208 , -10.8652462 ],
        [ -1.84903623,  -1.08458185, -10.84435978],
        [  2.00579583,  -1.08964491, -10.28965797],
        ...,
        [ -3.17886461,  -4.00037122,  -9.18725052],
        [  0.63541101,   5.69228601,   4.23193989],
        [ -4.29095806,   0.22884178,  -2.3275732 ]]),
 array([[-1.34829329e-03, -8.30541710e-04, -9.46022789e-03],
        [-1.48744725e-03, -7.61218281e-04, -9.45149041e-03],
        [ 1.77465076e-03, -1.03309394e-03, -9.05599574e-03],
        ...,
        [-2.67688103e-03, -3.48836339e-03, -8.33449933e-03],
        [ 6.23598267e-04,  5.03607911e-03,  3.46866549e-03],
        [-3.82186969e-03,  3.38224042e-05, -1.93300225e-03]]),
 [[-1.6923297148106442, -1.14332080176167, -10.865246200524389],
  [-1.849036226784979, -1.0845818548469972, -10.844359779320776],
  [2.00579582639953, -1.089644911792798, -10.28965797420603],
  [-1.6131891112683163, -0.9544878034859132, -10.878471755944311],
  [-1.766650686776434, -0.884331

In [43]:
generic(a=np.array(list(meshes_dict.values())[1]),b=np.array(list(meshes_dict.values())[0]))

846.078138567565

In [32]:
list(zip([1,2],range(2)))


[(1, 0), (2, 1)]

In [50]:
def compute_outliers(meshes_dict: Union[Dict[str,np.ndarray],List[np.ndarray]]):
    
    stats = []
    index_sum=None
    if isinstance(meshes_dict,list):
        num_of_points = len( meshes_dict[0] )
        working_dict = list(zip(range(num_of_points),meshes_dict))
        
    else:
        num_of_points = len( list(meshes_dict.values())[0] )
        working_dict = list(meshes_dict.items())
            
    for i in tqdm(range(num_of_points)): # i in range of num_of_points
        index = []
        points = []
        #print (i, working_dict)
        for key, mesh_v in working_dict:
            index.append(key)
            points.append(mesh_v[i])
        
        df = pd.DataFrame( data = points, index=index, columns=['x','y','z'])
        
        indexes = (np.abs(sci_stats.zscore(df)) < 2).all(axis=1)
        
        if index_sum is None:
            index_sum = (~indexes).astype(int)
        else:
            index_sum += (~indexes).astype(int)
        df = df[indexes]
        stats.append(df.describe())
    
    return stats, index_sum

stats,index_sum = compute_outliers(aligned_meshes)

  0%|          | 0/25680 [00:00<?, ?it/s]

In [6]:
len(aligned_meshes)

NameError: name 'aligned_meshes' is not defined

In [20]:
import scipy
scipy.__version__

'1.7.1'

In [5]:
sorted(index_sum.items(), key=lambda x:x[1])

[('wt_M_82Z-10911', 0),
 ('wt_M_82Z-CF279', 0),
 ('wt_F_82Z-06658', 0),
 ('wt_M_82Z-02973', 0),
 ('wt_M_82Z-11327', 0),
 ('wt_M_82Z-10878', 0),
 ('wt_F_82Z-10879', 0),
 ('wt_M_82Z-07309', 0),
 ('wt_F_82Z-07353', 0),
 ('wt_M_82Z-CG256', 0),
 ('wt_F_82Z-CH102', 0),
 ('wt_M_82Z-08858', 0),
 ('wt_F_82Z-CG092', 0),
 ('wt_F_82Z-06731', 0),
 ('wt_F_82Z-07750', 0),
 ('wt_M_82Z-CH144', 0),
 ('wt_M_82Z-07564', 0),
 ('wt_M_82Z-08685', 0),
 ('wt_F_82Z-CF025', 0),
 ('wt_M_82Z-06656', 0),
 ('wt_M_82Z-11639', 0),
 ('wt_M_82Z-CH035', 0),
 ('wt_F_82Z-09039', 0),
 ('wt_F_82Z-07311', 0),
 ('wt_F_82Z-CH156', 0),
 ('wt_M_82Z-11209', 0),
 ('wt_F_82Z-08624', 0),
 ('wt_M_82Z-10874', 0),
 ('wt_M_82Z-CG911', 0),
 ('wt_M_82Z-CH093', 0),
 ('wt_M_82Z-CF409', 0),
 ('wt_F_82Z-CE775', 0),
 ('wt_F_83Z-CE881', 0),
 ('wt_M_82Z-11985', 0),
 ('wt_F_82Z-07603', 0),
 ('wt_M_82Z-09591', 0),
 ('wt_M_82Z-CE568', 0),
 ('wt_M_82Z-07749', 0),
 ('wt_F_82Z-CG993', 0),
 ('wt_M_82Z-09738', 0),
 ('wt_M_82Z-07736', 0),
 ('wt_F_82Z-0939

In [6]:
len(sorted(index_sum.items(), key=lambda x:x[1]))

761

In [12]:
( path/  "affine_head.mat").remove

AttributeError: 'PosixPath' object has no attribute 'remove'

In [13]:
len(meshes_dict_copy)

406